# Volume 1. Basic Assembly Calculus

Question: what is the smallest useful assembly-calculus experiment?

This notebook is a visual first contact. You will build a tiny workbench with two stimuli, two source areas, and one target area. Each stimulus forms a sparse constellation of active winners. Then `merge` forms a new assembly in a target area.

The drawings are not anatomy. They are maps of neuron indices, useful for building intuition before moving to more formal overlap and readout diagnostics.

In [ ]:
from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import merge, project
from neural_assemblies.viz import plot_assemblies, plot_projection_flow

The whole experiment has one shape: two inputs form source assemblies, and the two source assemblies jointly drive a target assembly.

In [ ]:
plot_projection_flow(
    stimulus_labels=["stimulus s1", "stimulus s2"],
    source_labels=["area A1", "area A2"],
    target_label="merged area B",
);

Set up the bench. `N` is the number of neurons in each area, `K` is the number of winners, and `BETA` controls Hebbian strengthening after co-activation. The seed keeps the tour repeatable.

In [ ]:
N = 5_000
K = 80
BETA = 0.08
ROUNDS = 8

brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
brain.add_stimulus("s1", K)
brain.add_stimulus("s2", K)
brain.add_area("A1", n=N, k=K, beta=BETA)
brain.add_area("A2", n=N, k=K, beta=BETA)
brain.add_area("B", n=N, k=K, beta=BETA)

`project` lets a stimulus form a stable assembly in an area. Here `s1` writes into `A1`, and `s2` writes into `A2`.

In [ ]:
a1 = project(brain, "s1", "A1", rounds=ROUNDS)
a2 = project(brain, "s2", "A2", rounds=ROUNDS)

print(f"Source assembly sizes: {len(a1)}, {len(a2)}")
print(f"Source areas: {a1.area}, {a2.area}")

Visualize each assembly as active dots on a square canvas. The dots are winner neuron indices. They are sparse by design: only `K` out of `N` neurons are active.

In [ ]:
fig, axes = plot_assemblies(
    [a1, a2],
    n=N,
    titles=["A1 from stimulus s1", "A2 from stimulus s2"],
    colors=["#4d9de0", "#e15554"],
)
fig.suptitle("Two source assemblies", y=1.05)
fig

Checkpoint: a projection gives you a named sparse trace in a named area. You can inspect its size, area, and winners before claiming anything about what it represents.

In [ ]:
merged = merge(brain, "A1", "A2", "B", rounds=5)

print(f"Merged assembly size: {len(merged)}")
print(f"Merged assembly area: {merged.area}")

Now compare the three areas visually. The merged assembly is not the same object as either source assembly; it is a new sparse trace in area `B`, formed by the joint drive from `A1` and `A2`.

In [ ]:
fig, axes = plot_assemblies(
    [a1, a2, merged],
    n=N,
    titles=["source A1", "source A2", "merged target B"],
    colors=["#4d9de0", "#e15554", "#59a14f"],
)
fig.suptitle("Sparse assemblies as inspectable traces", y=1.05)
fig

Try next: change `K` from `80` to `40` or `120`, rerun the notebook, and watch the constellations become sparser or denser. Then move to the readout notebook, where overlap becomes the main diagnostic for comparing assemblies in the same area.